# P6｜Dataset-Balanced Sampler

**Pipeline ID：P6**  
**研究问题：** 在 P1 的 encoder、native heads、loss 与预算不变时，只把 source-proportional batch schedule 改为 dataset-balanced schedule，能否改善 worst-dataset/native retention？  
**Role：** source-balance one-axis comparator。  
**状态：Design / Not Ready。**

## Verified Contract / Proposed Method / HOLD

**Verified Contract**
- `baseline.four_dataset_frozen_encoder.train._source_batches` 已有 source-proportional 与 dataset-balanced 的可审计调度参考。
- dataset balance 与 within-task class/tail balance 是不同问题；P6 不复用 support-aware cRT 作为同一改变。

**Proposed Method**
- 复制 P1 的 AST、adapter、native heads、loss、trainable scope、总 optimizer steps、seed 与 selection。
- 唯一变化：每个 epoch/source 的 batch exposure 改为冻结的 dataset-balanced schedule；记录重复/采样权重与有效样本数。

**HOLD**
- balance 定义、epoch length、replacement、task-within-source routing 与 matched-compute 规则待冻结。
- class reweighting、cRT、eligibility objective、router/MoE 均不属于 P6。

| 组件 | P6 recipe |
|---|---|
| Model/loss | 与 P1 相同 |
| Sampler comparator | source-proportional → dataset-balanced |
| Compute | optimizer steps 与有效预算必须匹配 |
| Reporting | per-dataset/native task + worst-dataset；不合并 raw Score |


## 四数据集 native contract

ICBHI cycle flat4 `[B,4]`；SPRSound event binary `[B,2]`/raw7 `[B,7]`；HF 15-s recording observed-positive native heads；KAUH recording raw9 `[B,9]`。split/grouping、SPRSound terminal label join、HF gap omission、KAUH P-number/B-D-E grouping 与 shared/diagnosis HOLD 均不因 sampler 改变。

In [ ]:
import os
from pathlib import Path

PIPELINE_ID = "P6"
NOTEBOOK = Path("reproduce/P6_dataset_balanced_sampler.ipynb")
STATUS = "Design / Not Ready"
PROJECT_ROOT = Path.cwd() if Path.cwd().name != "reproduce" else Path.cwd().parent
DATASET_ROOT = Path(os.environ.get("ACOUSTIC_DATA_ROOT", "dataset/raw"))
CONFIG = Path("experiments/P6_dataset_balanced_sampler.yaml")
APPROVAL = Path("result/approvals/P6_execution_authorization.json")
assert NOTEBOOK.name.startswith(f"{PIPELINE_ID}_") and STATUS == "Design / Not Ready"


## Provenance / scope / budget / seed / selection / gates

P1 checkpoint/config hash、sampler schedule、replacement policy、epoch length、matched optimizer-step budget、seed、validation selection 与 worst-dataset go/no-go 必须在 outer/test 前冻结。preflight 必须验证每 source/task exposure receipt、missing-label omission、同一 group 不跨 split、P1 parity、本地 smoke 和 independent verifier；服务器仍需另行授权。

In [ ]:
required = {"config": PROJECT_ROOT / CONFIG, "approval": PROJECT_ROOT / APPROVAL}
EXECUTION_ALLOWED = False
dry_run_plan = {"pipeline_id": PIPELINE_ID, "backend_candidate": "baseline.four_dataset_frozen_encoder", "comparison": "dataset-balanced minus source-proportional; sampler only", "missing_gates": [k for k, v in required.items() if not v.is_file()]}
assert not EXECUTION_ALLOWED
dry_run_plan


## Outputs / receipt schema / claim boundary

未来 receipt：per-epoch source/task draw counts、replacement 与 duplicate rates、matched compute、native metrics/per-class/worst-dataset、group leakage audit、label-free outer lineage、verification 与 decision。

**Claim boundary：** 只评估 source exposure balance；不等于解决 class imbalance、tail objective 或 domain generalization。

**Test Result=Not run**  
**Decision：Not evaluated；Design / Not Ready。**